# ANP + Depth-aware — Hyperparameter Tuning

Notebook này giữ cấu trúc phần 1–2 của `notebook.ipynb`, dùng Optuna ở phần 3 để tìm cấu hình tốt và dùng các fold độc lập ở phần 4 để kiểm chứng cấu hình đã chọn.

# 1. Load Repo

In [ ]:
from pathlib import Path

if Path("/kaggle/working").exists():
    %cd /kaggle/working
    if not Path("/kaggle/working/well-log-imputation").exists():
        !git clone https://github.com/tranminhduc9/well-log-imputation.git /kaggle/working/well-log-imputation
    %cd /kaggle/working/well-log-imputation
else:
    print("Đang dùng repository hiện tại:", Path.cwd())

# 2. Requirements

In [ ]:
!cat requirements.txt
!python -m pip install -r requirements.txt
!python -m pip install "optuna>=4,<5"

# 3. Tinh chỉnh hyperparameter

Quy trình mặc định:

- Nested cross-validation rút gọn trên 3 outer folds (`2, 3, 4`).
- Trong mỗi outer fold, các giếng của outer-train được tách tiếp thành inner-train/inner-validation theo metadata giếng.
- Optuna chỉ nhìn inner-validation; outer-test tuyệt đối không dùng cho early stopping hoặc chọn tham số.
- Objective là macro-MAE của `single.1`, `block.20`, `block.100`, `profile`.
- Đánh giá cuối dùng 2 model seeds × 3 missing-mask seeds; cấu hình mặc định là đối chứng paired.

In [ ]:
import gc
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F

from main import get_dataset_dict, loading_data
from models.attention_neural_process import AttentionNeuralProcess
from models.attentive_neural_process import StandardAttentiveNeuralProcess

sns.set_theme(style="whitegrid", context="notebook")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

## 3.1. Cấu hình thí nghiệm

In [ ]:
PROJECT_ROOT = Path.cwd()
matches = list(Path("/kaggle/input").rglob("geolink_fold_0_well_log_sliced_train.npy")) if Path("/kaggle/input").exists() else []
local_data = PROJECT_ROOT / "imputation-processed-datasets"
DATA_DIR = matches[0].parent if matches else local_data
assert DATA_DIR.exists(), f"Không tìm thấy dataset tại {DATA_DIR}"

OUTPUT_DIR = Path("/kaggle/working/output/anp_tuning") if Path("/kaggle/working").exists() else PROJECT_ROOT / "output" / "anp_tuning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "geolink"
LOG_NAMES = ["GR", "DTC", "RHOB", "NPHI"]
N_STEPS = 256
N_FEATURES = len(LOG_NAMES)
OUTER_FOLDS = [2, 3, 4]
INNER_VAL_FRACTION = 0.15
N_TRIALS_PER_OUTER = 15
TUNING_EPOCHS = 100
TUNING_PATIENCE = 15
CONFIRM_EPOCHS = 250
CONFIRM_PATIENCE = 35
MODEL_SEEDS = [31415, 27182]
MASK_SEEDS = [91205, 91206, 91207]
MASK_SEED = 91205
RUN_CONFIRMATION = True
RUN_ABLATION = True

DEFAULT_PARAMS = {
    "hidden_dim": 128,
    "latent_dim": 32,
    "n_heads": 4,
    "initial_depth_scale": 0.2,
    "dropout": 0.1,
    "learning_rate": 3e-4,
    "weight_decay": 1e-5,
    "kl_weight": 3e-3,
    "kl_warmup_epochs": 40,
    "observed_loss_weight": 0.1,
    "batch_size": 32,
}

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 3.2. Tạo missing mask cố định

Mask được tạo một lần cho mỗi fold rồi cache trong RAM. `block` nhận danh sách một phần tử (`[20]`, `[100]`) để bảo đảm đúng chiều dài block.

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def concatenate_datasets(datasets):
    keys = set.intersection(*(set(dataset) for dataset in datasets))
    return {key: np.concatenate([dataset[key] for dataset in datasets], axis=0) for key in keys}


def make_evaluation_sets(array, seed):
    np.random.seed(seed)
    return {
        "single.1": get_dataset_dict(array, "single", n_points=1, fill=True, do_copy=True),
        "block.20": get_dataset_dict(array, "block", b_size=[20], fill=True, do_copy=True),
        "block.100": get_dataset_dict(array, "block", b_size=[100], fill=True, do_copy=True),
        "profile": get_dataset_dict(array, "profile", profile=None, fill=True, do_copy=True),
    }


FOLD_CACHE = {}


def prepare_outer_fold(fold: int):
    if fold in FOLD_CACHE:
        return FOLD_CACHE[fold]

    outer_train, outer_test = loading_data(DATASET_NAME, str(DATA_DIR), fold)
    if outer_train.shape[1:] != (N_STEPS, N_FEATURES):
        raise ValueError(f"Fold {fold} có shape {outer_train.shape}; kiểm tra N_STEPS và LOG_NAMES")

    metadata_path = DATA_DIR / f"{DATASET_NAME}_fold_{fold}_well_log_slices_meta_train.json"
    assert metadata_path.exists(), f"Thiếu metadata theo well: {metadata_path}"
    with open(metadata_path, encoding="utf-8") as file:
        slice_metadata = json.load(file)
    well_ids = np.asarray([str(row[0]) for row in slice_metadata])
    assert len(well_ids) == len(outer_train), "Metadata không khớp số sequence train"

    unique_wells = np.unique(well_ids)
    rng = np.random.default_rng(MASK_SEED + fold)
    shuffled_wells = rng.permutation(unique_wells)
    n_inner_val = max(1, int(round(INNER_VAL_FRACTION * len(shuffled_wells))))
    inner_val_wells = set(shuffled_wells[:n_inner_val])
    inner_val_mask = np.asarray([well in inner_val_wells for well in well_ids])
    inner_train_array = outer_train[~inner_val_mask]
    inner_val_array = outer_train[inner_val_mask]

    tuning_eval_sets = make_evaluation_sets(inner_val_array, MASK_SEED + 1000 + fold)
    inner_validation_set = concatenate_datasets(list(tuning_eval_sets.values()))
    FOLD_CACHE[fold] = {
        "inner_train_array": inner_train_array,
        "inner_validation": inner_validation_set,
        "tuning_eval_sets": tuning_eval_sets,
        "outer_test_array": outer_test,
        "n_train_wells": len(unique_wells) - n_inner_val,
        "n_val_wells": n_inner_val,
    }
    return FOLD_CACHE[fold]


for fold in OUTER_FOLDS:
    prepared = prepare_outer_fold(fold)
    print(f"Fold {fold}: {prepared['n_train_wells']} inner-train wells, {prepared['n_val_wells']} inner-validation wells")

## 3.3. Hàm huấn luyện và đánh giá

In [ ]:
Z_90 = 1.6448536269514722


def build_model(params, epochs, patience, output_name, model_class=AttentionNeuralProcess):
    return model_class(
        n_steps=N_STEPS,
        n_features=N_FEATURES,
        hidden_dim=int(params["hidden_dim"]),
        latent_dim=int(params["latent_dim"]),
        n_heads=int(params["n_heads"]),
        initial_depth_scale=float(params["initial_depth_scale"]),
        dropout=float(params["dropout"]),
        learning_rate=float(params["learning_rate"]),
        weight_decay=float(params["weight_decay"]),
        kl_weight=float(params["kl_weight"]),
        kl_warmup_epochs=int(params["kl_warmup_epochs"]),
        observed_loss_weight=float(params["observed_loss_weight"]),
        batch_size=int(params["batch_size"]),
        epochs=epochs,
        patience=patience,
        min_scale=0.03,
        max_scale=3.0,
        prediction_samples=16,
        device=DEVICE,
        saving_path=OUTPUT_DIR / output_name,
    )


def evaluate_model(model, evaluation_sets, fold, label, model_seed, mask_seed):
    rows = []
    for pattern, dataset in evaluation_sets.items():
        prediction = model.predict(dataset)
        mask = dataset["indicating_mask"].astype(bool)
        target = dataset["X_intact"]
        point = prediction["imputation"]
        lower = prediction["lower"]
        upper = prediction["upper"]
        for feature, log_name in [(None, "ALL")] + list(enumerate(LOG_NAMES)):
            selected = mask if feature is None else mask[:, :, feature]
            truth = target[selected] if feature is None else target[:, :, feature][selected]
            estimate = point[selected] if feature is None else point[:, :, feature][selected]
            lo = lower[selected] if feature is None else lower[:, :, feature][selected]
            hi = upper[selected] if feature is None else upper[:, :, feature][selected]
            finite = np.isfinite(truth) & np.isfinite(estimate) & np.isfinite(lo) & np.isfinite(hi)
            if not np.any(finite):
                continue
            truth, estimate, lo, hi = truth[finite], estimate[finite], lo[finite], hi[finite]
            sigma = np.maximum((hi - lo) / (2 * Z_90), 1e-6)
            rows.append({
                "label": label, "fold": fold, "model_seed": model_seed, "mask_seed": mask_seed,
                "pattern": pattern, "log": log_name,
                "mae": float(np.mean(np.abs(estimate - truth))),
                "rmse": float(np.sqrt(np.mean((estimate - truth) ** 2))),
                "picp": float(np.mean((truth >= lo) & (truth <= hi))),
                "mpiw": float(np.mean(hi - lo)),
                "nll": float(np.mean(0.5 * np.log(2 * np.pi * sigma**2) + 0.5 * ((truth - estimate) / sigma) ** 2)),
            })
    return rows


def train_model(params, fold, epochs, patience, label, model_seed, model_class=AttentionNeuralProcess):
    seed_everything(model_seed + fold)
    prepared = prepare_outer_fold(fold)
    np.random.seed(MASK_SEED + model_seed + fold)
    inner_train_set = get_dataset_dict(
        prepared["inner_train_array"], "rand", fill=False, do_copy=True
    )
    model = build_model(params, epochs, patience, f"{label}_fold_{fold}", model_class)
    model.fit(inner_train_set, prepared["inner_validation"])
    depth_scales = (
        F.softplus(model.attention.raw_depth_scale).detach().cpu().numpy().tolist()
        if hasattr(model.attention, "raw_depth_scale") else None
    )
    return model, depth_scales


def train_and_evaluate_inner(params, fold, epochs, patience, label, model_seed):
    model, depth_scales = train_model(params, fold, epochs, patience, label, model_seed)
    prepared = prepare_outer_fold(fold)
    rows = evaluate_model(
        model, prepared["tuning_eval_sets"], fold, label, model_seed, MASK_SEED
    )
    return model, rows, depth_scales

## 3.4. Không gian tìm kiếm Optuna

Các tham số uncertainty (`min_scale`, `max_scale`, `prediction_samples`) được giữ cố định ở vòng đầu để giới hạn chi phí.

In [ ]:
TUNED_PARAM_NAMES = [
    "hidden_dim", "latent_dim", "initial_depth_scale", "dropout",
    "learning_rate", "kl_weight", "observed_loss_weight",
]


def suggest_params(trial):
    suggested = {
        "hidden_dim": trial.suggest_categorical("hidden_dim", [64, 128, 256]),
        "latent_dim": trial.suggest_categorical("latent_dim", [16, 32, 64]),
        "initial_depth_scale": trial.suggest_float("initial_depth_scale", 0.03, 1.0, log=True),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 3e-4, log=True),
        "kl_weight": trial.suggest_float("kl_weight", 1e-4, 3e-2, log=True),
        "observed_loss_weight": trial.suggest_categorical("observed_loss_weight", [0.0, 0.03, 0.1, 0.3]),
    }
    return {**DEFAULT_PARAMS, **suggested}


def complete_params(partial_params):
    return {**DEFAULT_PARAMS, **partial_params}


def make_objective(outer_fold):
    def objective(trial):
        params = suggest_params(trial)
        try:
            model, rows, scales = train_and_evaluate_inner(
                params, outer_fold, TUNING_EPOCHS, TUNING_PATIENCE,
                f"outer_{outer_fold}_trial_{trial.number}", MODEL_SEEDS[0],
            )
            overall = [row["mae"] for row in rows if row["log"] == "ALL"]
            score = float(np.mean(overall))
            trial.set_user_attr("metrics", rows)
            trial.set_user_attr("learned_depth_scales", scales)
            trial.set_user_attr("outer_fold", outer_fold)
            del model
            return score
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return objective

## 3.5. Chạy study

In [ ]:
STORAGE = f"sqlite:///{(OUTPUT_DIR / 'optuna_study.db').as_posix()}"
STUDIES = {}
BEST_PARAMS = {}
default_search_params = {key: DEFAULT_PARAMS[key] for key in TUNED_PARAM_NAMES}

for outer_fold in OUTER_FOLDS:
    study_name = f"anp_depth_aware_{DATASET_NAME}_outer_{outer_fold}_v2"
    study = optuna.create_study(
        study_name=study_name, storage=STORAGE, direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=MASK_SEED + outer_fold, multivariate=True),
        load_if_exists=True,
    )
    if len(study.trials) == 0:
        study.enqueue_trial(default_search_params)
    remaining = max(0, N_TRIALS_PER_OUTER - len(study.trials))
    print(f"Outer fold {outer_fold}: đã có {len(study.trials)} trial, chạy thêm {remaining}.")
    if remaining:
        study.optimize(make_objective(outer_fold), n_trials=remaining, gc_after_trial=True)
    STUDIES[outer_fold] = study
    BEST_PARAMS[outer_fold] = complete_params(study.best_params)

with open(OUTPUT_DIR / "best_params.json", "w", encoding="utf-8") as file:
    json.dump(BEST_PARAMS, file, indent=2, ensure_ascii=False)

for fold, study in STUDIES.items():
    print(f"Outer fold {fold}: best inner-validation MAE = {study.best_value:.6f}")
    print(json.dumps(BEST_PARAMS[fold], indent=2))

# 4. Visualize và kiểm chứng

Phần này trả lời bốn câu hỏi: study đã hội tụ chưa, tham số nào ảnh hưởng mạnh, cấu hình tối ưu có tốt hơn mặc định trên inner-validation không, và cải thiện có lặp lại trên outer-test hay không. PICP/MPIW là metrics chính cho uncertainty; NLL bên dưới là Gaussian moment-matched approximation. Vì input chưa chứa độ sâu vật lý, `depth` của model là vị trí mẫu tương đối chuẩn hóa `[-1, 1]` và phải được mô tả đúng như vậy trong paper.

## 4.1. Lịch sử tối ưu và độ quan trọng tham số

In [ ]:
history_parts, importance_parts, top_rows = [], [], []
for fold, fold_study in STUDIES.items():
    completed = [trial for trial in fold_study.trials if trial.state == optuna.trial.TrialState.COMPLETE]
    part = pd.DataFrame({"trial": [t.number for t in completed], "objective": [t.value for t in completed]})
    part = part.sort_values("trial")
    part["best_so_far"] = part["objective"].cummin()
    part["outer_fold"] = fold
    history_parts.append(part)
    try:
        importance_parts.append(pd.Series(optuna.importance.get_param_importances(fold_study), name=fold))
    except Exception:
        pass
    top_rows.append({"outer_fold": fold, "best_mae": fold_study.best_value, **fold_study.best_params})
history = pd.concat(history_parts, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for fold, part in history.groupby("outer_fold"):
    axes[0].plot(part["trial"], part["best_so_far"], marker="o", linewidth=2, label=f"Outer fold {fold}")
axes[0].set(title="Lịch sử tối ưu", xlabel="Trial", ylabel="Macro validation MAE")
axes[0].legend()

try:
    importance_series = pd.DataFrame(importance_parts).fillna(0).mean().sort_values()
    importance_series.plot.barh(ax=axes[1], color="#0072B2")
    axes[1].set(title="Độ quan trọng của hyperparameter", xlabel="Importance")
except Exception as error:
    axes[1].text(0.5, 0.5, f"Chưa tính được importance:\n{error}", ha="center", va="center")
    axes[1].set_axis_off()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "optimization_history_and_importance.png", dpi=160, bbox_inches="tight")
plt.show()

display(pd.DataFrame(top_rows).round(6))

## 4.2. So sánh Default và Optimized trên folds tuning

In [ ]:
def metrics_frame(trial, label):
    frame = pd.DataFrame(trial.user_attrs.get("metrics", []))
    if not frame.empty:
        frame["label"] = label
    return frame


def is_default_trial(trial):
    return trial.state == optuna.trial.TrialState.COMPLETE and all(
        trial.params.get(key) == DEFAULT_PARAMS[key] for key in TUNED_PARAM_NAMES
    )


comparison_parts = []
for fold, fold_study in STUDIES.items():
    default_candidates = [trial for trial in fold_study.trials if is_default_trial(trial)]
    assert default_candidates, f"Không tìm thấy baseline mặc định cho outer fold {fold}."
    comparison_parts.extend([
        metrics_frame(default_candidates[0], "Default"),
        metrics_frame(fold_study.best_trial, "Optimized"),
    ])
tuning_comparison = pd.concat(comparison_parts, ignore_index=True)
tuning_overall = tuning_comparison[tuning_comparison["log"] == "ALL"]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=tuning_overall, x="pattern", y="mae", hue="label", errorbar="sd", ax=axes[0])
axes[0].set(title="MAE trên folds tuning", xlabel="Missing pattern", ylabel="MAE")
sns.boxplot(data=tuning_overall, x="label", y="mae", hue="label", legend=False, ax=axes[1])
sns.stripplot(data=tuning_overall, x="label", y="mae", color="black", alpha=0.55, ax=axes[1])
axes[1].set(title="Độ ổn định qua fold và pattern", xlabel="Cấu hình", ylabel="MAE")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "tuning_default_vs_optimized.png", dpi=160, bbox_inches="tight")
plt.show()

## 4.3. Xác nhận trên folds độc lập

Mỗi outer-test fold chưa từng được dùng bởi study tương ứng. Default và Optimized được huấn luyện với cùng inner split, model seeds và missing masks, tạo so sánh paired công bằng. Có thể đặt `RUN_CONFIRMATION=False` để đọc lại CSV đã chạy.

In [ ]:
confirmation_path = OUTPUT_DIR / "confirmation_metrics.csv"
history_path = OUTPUT_DIR / "training_history.csv"
confirmation_rows = []
history_rows = []

if RUN_CONFIRMATION:
    for fold in OUTER_FOLDS:
        prepared = prepare_outer_fold(fold)
        outer_eval_sets = {
            mask_seed: make_evaluation_sets(prepared["outer_test_array"], mask_seed + 10000 + fold)
            for mask_seed in MASK_SEEDS
        }
        runs = [
            ("Default", DEFAULT_PARAMS, AttentionNeuralProcess, MODEL_SEEDS),
            ("Optimized", BEST_PARAMS[fold], AttentionNeuralProcess, MODEL_SEEDS),
        ]
        if RUN_ABLATION:
            runs.append(("ANP standard", BEST_PARAMS[fold], StandardAttentiveNeuralProcess, MODEL_SEEDS[:1]))
        for label, params, model_class, seeds in runs:
            for model_seed in seeds:
                print(f"\nOuter-test: {label}, fold {fold}, model seed {model_seed}")
                model, scales = train_model(
                    params, fold, CONFIRM_EPOCHS, CONFIRM_PATIENCE, label.lower(), model_seed, model_class
                )
                history_rows.extend([
                    {"label": label, "fold": fold, "model_seed": model_seed, **epoch_metrics}
                    for epoch_metrics in model.training_history
                ])
                for mask_seed, evaluation_sets in outer_eval_sets.items():
                    confirmation_rows.extend(evaluate_model(
                        model, evaluation_sets, fold, label, model_seed, mask_seed
                    ))
                if label == "Optimized":
                    model.save(OUTPUT_DIR / f"optimized_fold_{fold}_seed_{model_seed}.pt")
                    with open(OUTPUT_DIR / f"depth_scales_fold_{fold}_seed_{model_seed}.json", "w") as file:
                        json.dump(scales, file, indent=2)
                del model
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    confirmation = pd.DataFrame(confirmation_rows)
    confirmation.to_csv(confirmation_path, index=False)
    training_history = pd.DataFrame(history_rows)
    training_history.to_csv(history_path, index=False)
else:
    assert confirmation_path.exists(), f"Chưa có {confirmation_path}"
    assert history_path.exists(), f"Chưa có {history_path}"
    confirmation = pd.read_csv(confirmation_path)
    training_history = pd.read_csv(history_path)

display(confirmation.groupby(["label", "pattern"])[["mae", "rmse", "picp", "mpiw", "nll"]].agg(["mean", "std"]).round(4))

## 4.4. Biểu đồ kiểm chứng cuối

In [ ]:
overall = confirmation[confirmation["log"] == "ALL"].copy()
per_log = confirmation[confirmation["log"] != "ALL"].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
sns.barplot(data=overall, x="pattern", y="mae", hue="label", errorbar="sd", ax=axes[0, 0])
axes[0, 0].set(title="MAE trên folds độc lập", xlabel="Missing pattern", ylabel="MAE")

sns.boxplot(data=overall, x="pattern", y="mae", hue="label", ax=axes[0, 1])
axes[0, 1].set(title="Phân bố MAE giữa các fold", xlabel="Missing pattern", ylabel="MAE")

sns.barplot(data=overall, x="pattern", y="picp", hue="label", errorbar="sd", ax=axes[1, 0])
axes[1, 0].axhline(0.90, color="black", linestyle="--", linewidth=1.5, label="Nominal 90%")
axes[1, 0].set(title="Coverage của khoảng dự báo 90%", xlabel="Missing pattern", ylabel="PICP", ylim=(0, 1.05))

sns.barplot(data=overall, x="pattern", y="mpiw", hue="label", errorbar="sd", ax=axes[1, 1])
axes[1, 1].set(title="Độ rộng khoảng dự báo", xlabel="Missing pattern", ylabel="MPIW")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confirmation_summary.png", dpi=170, bbox_inches="tight")
plt.show()

mae_table = per_log.groupby(["label", "pattern", "log"])["mae"].mean().unstack("label")
mae_table["improvement_%"] = 100 * (mae_table["Default"] - mae_table["Optimized"]) / mae_table["Default"]
improvement = mae_table["improvement_%"].unstack("log").reindex(columns=LOG_NAMES)

plt.figure(figsize=(10, 5))
sns.heatmap(improvement, annot=True, fmt=".1f", center=0, cmap="RdYlGn", cbar_kws={"label": "MAE improvement (%)"})
plt.title("Mức cải thiện của Optimized so với Default theo từng log")
plt.xlabel("Well log")
plt.ylabel("Missing pattern")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confirmation_per_log_improvement.png", dpi=170, bbox_inches="tight")
plt.show()

pair_keys = ["fold", "model_seed", "mask_seed", "pattern"]
paired = overall.pivot_table(index=pair_keys, columns="label", values="mae").dropna().reset_index()
paired["mae_gain"] = paired["Default"] - paired["Optimized"]
fold_gain = paired.groupby("fold", as_index=False)["mae_gain"].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sns.barplot(data=fold_gain, x="fold", y="mae_gain", color="#009E73", ax=axes[0])
axes[0].axhline(0, color="black", linewidth=1.2)
axes[0].set(title="Lợi ích của tuning trên outer-test", xlabel="Outer fold", ylabel="MAE(Default) − MAE(Optimized)")

if "ANP standard" in set(overall["label"]):
    ablation_source = overall[overall["model_seed"] == MODEL_SEEDS[0]]
    ablation = ablation_source.pivot_table(index=pair_keys, columns="label", values="mae").dropna().reset_index()
    ablation["depth_gain"] = ablation["ANP standard"] - ablation["Optimized"]
    depth_fold_gain = ablation.groupby("fold", as_index=False)["depth_gain"].mean()
    sns.barplot(data=depth_fold_gain, x="fold", y="depth_gain", color="#0072B2", ax=axes[1])
    axes[1].axhline(0, color="black", linewidth=1.2)
    axes[1].set(title="Ablation của depth-aware attention", xlabel="Outer fold", ylabel="MAE(Standard ANP) − MAE(Depth-aware)")
else:
    axes[1].set_axis_off()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "paired_outer_fold_gain.png", dpi=170, bbox_inches="tight")
plt.show()

## 4.5. Bảng so sánh định lượng giữa các mô hình

Bảng này tổng hợp kết quả outer-test của Default depth-aware ANP, Optimized depth-aware ANP và Standard ANP. Seed, mask và pattern được trung bình hóa bên trong từng outer fold trước; vì vậy `std` là biến thiên giữa các outer fold, tránh pseudo-replication.

In [ ]:
metric_columns = ["mae", "rmse", "picp", "mpiw", "nll"]
fold_level_overall = overall.groupby(["label", "fold"], as_index=False)[metric_columns].mean()
fold_level_pattern = overall.groupby(["label", "pattern", "fold"], as_index=False)[metric_columns].mean()

overall_comparison = (
    fold_level_overall.groupby("label")
    .agg(
        mae_mean=("mae", "mean"), mae_std=("mae", "std"),
        rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"),
        picp_mean=("picp", "mean"), picp_std=("picp", "std"),
        mpiw_mean=("mpiw", "mean"), mpiw_std=("mpiw", "std"),
        nll_mean=("nll", "mean"), nll_std=("nll", "std"),
        n_outer_folds=("mae", "size"),
    )
    .sort_values("mae_mean")
)
pattern_comparison = (
    fold_level_pattern.groupby(["label", "pattern"])
    .agg(
        mae_mean=("mae", "mean"), mae_std=("mae", "std"),
        rmse_mean=("rmse", "mean"), picp_mean=("picp", "mean"),
        mpiw_mean=("mpiw", "mean"), nll_mean=("nll", "mean"),
        n_outer_folds=("mae", "size"),
    )
)

display(overall_comparison.round(5))
display(pattern_comparison.round(5))
overall_comparison.to_csv(OUTPUT_DIR / "model_comparison_overall.csv")
pattern_comparison.to_csv(OUTPUT_DIR / "model_comparison_by_pattern.csv")

## 4.6. So sánh paired và khoảng tin cậy 95%

Mỗi cặp dùng cùng outer fold, model seed, mask seed và missing pattern. Giá trị `mean_gain > 0` nghĩa là mô hình ở bên phải tốt hơn mô hình tham chiếu. Với PICP, notebook so sánh sai lệch tuyệt đối so với coverage mục tiêu 0.90. Khoảng tin cậy được cluster-bootstrap theo outer fold; đây là bằng chứng mô tả, không phải phép chứng minh global optimum.

In [ ]:
paired_source = overall.copy()
paired_source["coverage_error"] = (paired_source["picp"] - 0.90).abs()
PAIR_KEYS = ["fold", "model_seed", "mask_seed", "pattern"]
COMPARISONS = [
    ("Default → Optimized", "Default", "Optimized"),
    ("Standard ANP → Depth-aware", "ANP standard", "Optimized"),
]
METRICS_FOR_GAIN = {
    "MAE": "mae",
    "RMSE": "rmse",
    "Coverage error": "coverage_error",
    "NLL (approx.)": "nll",
    "MPIW": "mpiw",
}


def clustered_gain_summary(frame, reference, candidate, metric, bootstrap_seed=2026):
    selected = frame[frame["label"].isin([reference, candidate])]
    pivot = selected.pivot_table(index=PAIR_KEYS, columns="label", values=metric).dropna()
    if reference not in pivot or candidate not in pivot:
        return None
    gain = (pivot[reference] - pivot[candidate]).rename("gain").reset_index()
    fold_means = gain.groupby("fold")["gain"].mean().to_numpy()
    rng = np.random.default_rng(bootstrap_seed)
    bootstrap = rng.choice(
        fold_means, size=(10000, len(fold_means)), replace=True
    ).mean(axis=1)
    return {
        "mean_gain": float(gain["gain"].mean()),
        "ci_low": float(np.quantile(bootstrap, 0.025)),
        "ci_high": float(np.quantile(bootstrap, 0.975)),
        "paired_units": len(gain),
        "positive_units_%": 100 * float((gain["gain"] > 0).mean()),
        "positive_folds": int((fold_means > 0).sum()),
        "total_folds": len(fold_means),
    }


gain_rows = []
for comparison_name, reference, candidate in COMPARISONS:
    for metric_name, metric_column in METRICS_FOR_GAIN.items():
        result = clustered_gain_summary(
            paired_source, reference, candidate, metric_column,
            bootstrap_seed=2026 + len(gain_rows),
        )
        if result is not None:
            gain_rows.append({"comparison": comparison_name, "metric": metric_name, **result})
gain_summary = pd.DataFrame(gain_rows)
display(gain_summary.round(6))
gain_summary.to_csv(OUTPUT_DIR / "paired_model_gain_with_ci.csv", index=False)

metrics_to_plot = ["MAE", "RMSE", "Coverage error", "NLL (approx.)"]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for axis, metric_name in zip(axes.flat, metrics_to_plot):
    plot_data = gain_summary[gain_summary["metric"] == metric_name].reset_index(drop=True)
    positions = np.arange(len(plot_data))
    axis.errorbar(
        plot_data["mean_gain"], positions,
        xerr=[plot_data["mean_gain"] - plot_data["ci_low"], plot_data["ci_high"] - plot_data["mean_gain"]],
        fmt="o", capsize=5, color="#0072B2", markersize=7,
    )
    axis.axvline(0, color="black", linestyle="--", linewidth=1)
    axis.set_yticks(positions, plot_data["comparison"])
    axis.set(title=metric_name, xlabel="Paired gain (positive = improvement)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "paired_gain_confidence_intervals.png", dpi=170, bbox_inches="tight")
plt.show()

## 4.7. Liên kết từng bộ tham số với kết quả outer-test

Bảng dưới đây cho phép đánh giá từng bộ tham số được chọn ở mỗi outer fold. Không tự động gộp chúng thành một bộ duy nhất: hãy ưu tiên cấu hình có outer-test MAE/RMSE thấp, coverage gần 0.90 và kết quả ổn định giữa các fold. Bảng top-5 inner-validation trials giúp kiểm tra bộ tốt nhất có vượt trội thật sự hay chỉ chênh rất ít so với các cấu hình lân cận.

In [ ]:
config_rows = []
for fold in OUTER_FOLDS:
    optimized_fold = overall[(overall["label"] == "Optimized") & (overall["fold"] == fold)]
    default_fold = overall[(overall["label"] == "Default") & (overall["fold"] == fold)]
    params = BEST_PARAMS[fold]
    config_rows.append({
        "outer_fold": fold,
        "inner_best_mae": STUDIES[fold].best_value,
        "outer_mae": optimized_fold["mae"].mean(),
        "outer_mae_std": optimized_fold["mae"].std(),
        "mae_gain_vs_default": default_fold["mae"].mean() - optimized_fold["mae"].mean(),
        "outer_rmse": optimized_fold["rmse"].mean(),
        "outer_picp": optimized_fold["picp"].mean(),
        "coverage_error": abs(optimized_fold["picp"].mean() - 0.90),
        "outer_mpiw": optimized_fold["mpiw"].mean(),
        **params,
    })
config_evidence = pd.DataFrame(config_rows).set_index("outer_fold")
display(config_evidence.round(6))
config_evidence.to_csv(OUTPUT_DIR / "best_params_outer_test_evidence.csv")

top_trial_rows = []
for fold, fold_study in STUDIES.items():
    complete_trials = sorted(
        [trial for trial in fold_study.trials if trial.state == optuna.trial.TrialState.COMPLETE],
        key=lambda trial: trial.value,
    )[:5]
    for rank, trial in enumerate(complete_trials, start=1):
        top_trial_rows.append({
            "outer_fold": fold, "rank": rank, "inner_mae": trial.value, **trial.params
        })
top_trials = pd.DataFrame(top_trial_rows)
display(top_trials.round(6))
top_trials.to_csv(OUTPUT_DIR / "top5_params_per_outer_fold.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.scatterplot(
    data=config_evidence.reset_index(), x="inner_best_mae", y="outer_mae",
    hue="outer_fold", palette="viridis", s=130, ax=axes[0],
)
axes[0].set(title="Inner-validation và outer-test MAE", xlabel="Inner best MAE", ylabel="Outer-test MAE")

param_view = config_evidence[TUNED_PARAM_NAMES].copy()
param_normalized = (param_view - param_view.min()) / (param_view.max() - param_view.min()).replace(0, 1)
sns.heatmap(param_normalized, annot=param_view, fmt=".4g", cmap="Blues", cbar_kws={"label": "Min-max normalized"}, ax=axes[1])
axes[1].set(title="Độ ổn định của tham số được chọn", xlabel="Hyperparameter", ylabel="Outer fold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "parameter_sets_and_outer_evidence.png", dpi=170, bbox_inches="tight")
plt.show()

## 4.8. Tốc độ hội tụ theo epoch

Các đường biểu diễn trung bình qua outer folds và model seeds; vùng mờ là một độ lệch chuẩn. Vì early stopping làm số run còn hoạt động giảm ở các epoch cuối, bảng bên dưới cũng báo epoch tốt nhất và tổng số epoch thực chạy cho từng mô hình.

In [ ]:
assert not training_history.empty, "Training history trống; hãy chạy phần 4.3 trước."
training_history["epoch"] = training_history["epoch"].astype(int)

convergence_metrics = [
    ("train_loss", "Training loss"),
    ("val_loss", "Inner-validation loss"),
    ("val_mae", "Inner-validation MAE"),
    ("val_missing_nll", "Inner-validation missing NLL"),
    ("val_kl", "Inner-validation KL divergence"),
    ("learning_rate", "Learning rate"),
]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for axis, (metric, title) in zip(axes.flat, convergence_metrics):
    sns.lineplot(
        data=training_history, x="epoch", y=metric, hue="label",
        estimator="mean", errorbar="sd", linewidth=2, ax=axis,
    )
    axis.set(title=title, xlabel="Epoch", ylabel=metric)
    axis.grid(alpha=0.22)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "training_convergence.png", dpi=170, bbox_inches="tight")
plt.show()

run_keys = ["label", "fold", "model_seed"]
best_indices = training_history.groupby(run_keys)["val_mae"].idxmin()
best_epoch_runs = training_history.loc[best_indices, run_keys + ["epoch", "val_mae", "val_loss"]].copy()
best_epoch_runs = best_epoch_runs.rename(columns={"epoch": "best_epoch", "val_mae": "best_val_mae", "val_loss": "loss_at_best_epoch"})
epochs_run = training_history.groupby(run_keys, as_index=False)["epoch"].max().rename(columns={"epoch": "epochs_run"})
best_epoch_runs = best_epoch_runs.merge(epochs_run, on=run_keys, how="left")
convergence_summary = best_epoch_runs.groupby("label").agg(
    best_epoch_mean=("best_epoch", "mean"),
    best_epoch_std=("best_epoch", "std"),
    epochs_run_mean=("epochs_run", "mean"),
    epochs_run_std=("epochs_run", "std"),
    best_val_mae_mean=("best_val_mae", "mean"),
    best_val_mae_std=("best_val_mae", "std"),
    n_runs=("best_epoch", "size"),
)
display(best_epoch_runs.sort_values(run_keys).round(6))
display(convergence_summary.round(4))
best_epoch_runs.to_csv(OUTPUT_DIR / "best_epoch_per_run.csv", index=False)
convergence_summary.to_csv(OUTPUT_DIR / "convergence_summary.csv")